# dgm-eval on the four-channel FDGAN COCO PottedPlant samples

Same cell sequence as `dgm-eval_Clean_PottedPlant_all.ipynb` (your last dgm-eval run, 28 Dec 2025). Only the dataset cells changed.

**Run order:** cells 1–2, then cell 4 (package pins) → **Runtime ▸ Restart session** → continue from cell 5 onwards.

**Before you start:** put the four-channel generated set on Drive (a zip, a folder of PNGs, or the sampler's `.npz`) and write its path into the CONFIG cell.

dgm-eval argument order (from its README and `__main__.py`): `python -m dgm_eval <REAL_TRAIN> <GENERATED> --test_path <REAL_TEST>`.
The first path is the reference set, every further path is a generated set, `--test_path` is the held-out real set that FLS, FLS-Overfit, authpct and C_T need.


In [ ]:
%cd /content
!rm -rf dgm-eval
!git clone https://github.com/layer6ai-labs/dgm-eval.git
%cd dgm-eval


In [ ]:
!pip install --quiet --no-deps -e .


**Restart Session after the cell below** (Runtime ▸ Restart session), then continue from the version-check cell.


In [ ]:
# Core scientific stack (plays well on Colab Py3.12)
!pip install -q "numpy==2.2.6" "scipy==1.16.1" "pandas==2.2.2" "scikit-learn==1.7.2"

# Vision + image utils
!pip install -q "opencv-python==4.12.0.88" "pillow==10.4.0" "scikit-image==0.24.0"

# Encoders used by dgm-eval  (xformers deliberately NOT installed any more: the Oct 2026 Colab image ships
# PyTorch 2.11 / Python 3.13, the xformers wheel was built for PyTorch 2.10 / Python 3.10, and DINOv2 then crashes
# inside memory_efficient_attention. DINOv2 runs its plain attention instead; the features are the same.)
!pip install -q "open_clip_torch==2.19.0" "timm==0.9.16" "transformers==4.45.2"


In [ ]:
import numpy, pandas, sklearn, scipy, PIL, cv2, skimage
import torch, torchvision
import open_clip, timm, transformers

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("SciPy:", scipy.__version__)
print("Pillow:", PIL.__version__)
print("cv2:", cv2.__version__)
print("skimage:", skimage.__version__)
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("open_clip:", getattr(open_clip, "__version__", "ok"))
print("timm:", timm.__version__)
print("transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU")
print("All good.")


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

# create symbolic link
!ln -s /content/gdrive/My\ Drive/ /mydrive

# list gdrive content
!ls /mydrive


In [ ]:
!unzip -q /content/gdrive/MyDrive/PhD/COCO/PottedPlantTestSet2_k.zip -d /content/dgm-eval/


In [ ]:
!unzip -q /content/gdrive/MyDrive/PhD/COCO/COCO_PottedPlant_256sq_Img.zip -d /content/dgm-eval/


In [ ]:
!pwd


In [ ]:
%cd /content/dgm-eval


In [ ]:
# ==== keep TensorFlow out of the way ====================================================================
# open_clip_torch 2.19.0 pins protobuf<4, and Colab's TensorFlow (Python 3.13 image) needs protobuf>=5.26.
# `transformers` imports TensorFlow whenever it sees it installed, which is where the
# "cannot import name 'runtime_version' from 'google.protobuf'" crash came from. dgm-eval never uses TensorFlow,
# so we tell `transformers` to ignore it. The two variables are inherited by every !python command below.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
# XFORMERS_DISABLED=1 is read by the DINOv2 hub code (dinov2/layers/attention.py, swiglu_ffn.py, block.py):
# with it set, DINOv2 uses its ordinary PyTorch attention instead of xformers' memory_efficient_attention,
# which cannot run when the installed xformers wheel does not match Colab's PyTorch build.
os.environ["XFORMERS_DISABLED"] = "1"

# smoke test: this must print the encoder list (inception, dinov2, ...) and no traceback
!USE_TF=0 USE_TORCH=1 XFORMERS_DISABLED=1 python -c "import google.protobuf as p; print('protobuf', p.__version__); from dgm_eval.models import MODELS; print('dgm-eval import OK:', list(MODELS))"


In [ ]:
# ==== FALLBACKS - run ONLY if the smoke test above still shows a traceback ===============================
# F1: newer protobuf (overrides open_clip's old pin; open_clip only needs protobuf for text tokenisers, not images)
#!pip install -q "protobuf>=5.29,<7"
# F2: drop the unused data2vec encoder, which is the only thing in dgm-eval that imports `transformers`
#!sed -i 's/^from .data2vec import HuggingFaceTransformerEncoder/HuggingFaceTransformerEncoder = None  # disabled/' /content/dgm-eval/dgm_eval/models/__init__.py /content/dgm-eval/dgm_eval/models/load_encoder.py
# F3: if DINOv2 still dies inside xformers "memory_efficient_attention" (an old xformers wheel is installed in this
#     session), remove xformers altogether; DINOv2 then falls back to plain attention by itself
#!pip uninstall -y -q xformers
# then re-run the smoke test cell above


In [ ]:
# ==== CONFIG: the four-channel FDGAN generated set ===========================
GEN_SRC = "/content/gdrive/MyDrive/PATH/TO/FDGAN_4ch_generated.zip"   # <-- zip, folder of PNGs, or sampler .npz on Drive
TAG     = "FDGAN_4ch"                                                  # <-- short label used in all output file names

RESULTS_DIR = "/content/gdrive/MyDrive/PhD/COCO/FDGAN_4ch_metrics"    # txt + xlsx results go here
REAL_DIR = "/content/dgm-eval/COCO_PottedPlant_256sq_Img"              # unzipped above
TEST_DIR = "/content/dgm-eval/PottedPlantTestSet2_k"                   # unzipped above
GEN_DIR  = f"/content/dgm-eval/{TAG}"                                  # flat folder of generated images, built below
# =============================================================================
import os, re, glob, shutil, zipfile
import numpy as np
from PIL import Image
os.makedirs(RESULTS_DIR, exist_ok=True)

# dgm-eval reads only the files directly inside a folder (or treats sub-folders as classes), so the generated set
# must be ONE flat folder that contains sample images only: no masks, grids, overlays, json or npz side-files.
IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
SKIP = (r"mask", r"grid", r"overlay", r"_bb\b", r"bbox", r"meta", r"pair", r"debug", r"real")

def stage(src_root, dst):
    os.makedirs(dst, exist_ok=True); kept = skipped = 0; skipped_names = []
    for dp, dn, fn in os.walk(src_root):
        if "__MACOSX" in dp:
            continue
        for f in sorted(fn):
            if not f.lower().endswith(IMG_EXT) or f.startswith("._"):
                continue
            if any(re.search(p, f, re.I) for p in SKIP):
                skipped += 1; skipped_names.append(f); continue
            out = os.path.join(dst, f)
            if os.path.exists(out):                       # same name in two sub-folders
                out = os.path.join(dst, os.path.basename(dp) + "_" + f)
            shutil.copy2(os.path.join(dp, f), out); kept += 1
    return kept, skipped, skipped_names

if os.path.isdir(GEN_DIR) and os.listdir(GEN_DIR):
    print("already staged:", GEN_DIR)
elif GEN_SRC.lower().endswith(".npz"):
    arr = np.load(GEN_SRC)["arr_0"]; os.makedirs(GEN_DIR, exist_ok=True)
    for i, im in enumerate(arr):
        Image.fromarray(im).save(f"{GEN_DIR}/{i:06d}.png")
    print(f"npz -> {len(arr)} PNGs in {GEN_DIR}")
else:
    src = GEN_SRC
    if GEN_SRC.lower().endswith(".zip"):
        src = f"/content/dgm-eval/{TAG}_raw"; os.makedirs(src, exist_ok=True)
        with zipfile.ZipFile(GEN_SRC) as z:
            z.extractall(src)
    kept, skipped, names = stage(src, GEN_DIR)
    print(f"kept {kept} sample images, skipped {skipped} side-files -> {GEN_DIR}")
    if names:
        print("   skipped e.g.:", names[:8])
    if kept == 0:
        npzs = glob.glob(os.path.join(src, "**", "*.npz"), recursive=True)
        assert npzs, "No images and no npz found under GEN_SRC - check the path"
        arr = np.load(npzs[0])["arr_0"]
        for i, im in enumerate(arr):
            Image.fromarray(im).save(f"{GEN_DIR}/{i:06d}.png")
        print(f"no PNGs; used {npzs[0]} -> {len(arr)} PNGs")

def describe(d, name):
    fs = sorted(f for f in os.listdir(d) if f.lower().endswith(IMG_EXT))
    sizes = {}
    for f in fs[:300]:
        try:
            s = Image.open(os.path.join(d, f)).size; sizes[s] = sizes.get(s, 0) + 1
        except Exception:
            pass
    print(f"{name:10s} {len(fs):5d} images   sizes (first 300): {sizes}   e.g. {fs[:3]}")
    return len(fs)

n_real = describe(REAL_DIR, "real_train")
n_test = describe(TEST_DIR, "real_test")
n_gen  = describe(GEN_DIR,  "generated")


In [ ]:
import os
from PIL import Image

bad_files = []
for d in (GEN_DIR, REAL_DIR, TEST_DIR):
    for fname in sorted(os.listdir(d)):
        fpath = os.path.join(d, fname)
        if not os.path.isfile(fpath):
            continue
        try:
            with Image.open(fpath) as img:
                img.verify()  # check if it is a valid image
        except Exception as e:
            print("Bad file:", fpath, "->", e)
            bad_files.append(fpath)

print("\nTotal bad files:", len(bad_files))
for f in bad_files:                 # a corrupt file aborts dgm-eval, so remove it (same as the Dec 2025 notebook did)
    os.remove(f); print("Deleted:", f)


In [ ]:
# ============== RUN A: DINOv2 (the paper's primary encoder), documented argument order ==============
# real-train first, generated second, held-out real test via --test_path. nsample 3000 as in the Nov 2025 table runs
# (a set with fewer images is used whole). Metrics list = the one used for the paper's table runs.
!USE_TF=0 USE_TORCH=1 XFORMERS_DISABLED=1 python -m dgm_eval {REAL_DIR} {GEN_DIR} --test_path {TEST_DIR} \
                    --model dinov2 \
                    --metrics fd kd prdc fd-infinity aws is mmd authpct sw_approx fls fls_overfit ct ct_modified \
                    --device cuda \
                    --batch_size 128 \
                    --nsample 3000 \
                    --seed 42 \
                    --output_dir /content/dgm-eval/experiments_{TAG} \
                    2>&1 | tee {RESULTS_DIR}/{TAG}_dinov2.txt


In [ ]:
# ============== RUN B: Inception-V3 (gives the FID column) ==============
!USE_TF=0 USE_TORCH=1 XFORMERS_DISABLED=1 python -m dgm_eval {REAL_DIR} {GEN_DIR} --test_path {TEST_DIR} \
                    --model inception \
                    --metrics fd kd prdc fd-infinity is \
                    --device cuda \
                    --batch_size 128 \
                    --nsample 3000 \
                    --seed 42 \
                    --output_dir /content/dgm-eval/experiments_{TAG} \
                    2>&1 | tee {RESULTS_DIR}/{TAG}_inception.txt


In [ ]:
# ============== RUN C (comparison only): the order used in the Nov 2025 table notebooks, generated set FIRST ==============
# FD, KD, FID are identical either way. precision<->recall, density<->coverage, FLS, FLS-Overfit, authpct and C_T are not.
# This run exists only so the new row can be compared like-for-like with the columns already in the table.
!USE_TF=0 USE_TORCH=1 XFORMERS_DISABLED=1 python -m dgm_eval {GEN_DIR} {REAL_DIR} --test_path {TEST_DIR} \
                    --model dinov2 \
                    --metrics fd kd prdc fd-infinity authpct fls fls_overfit ct ct_modified \
                    --device cuda \
                    --batch_size 128 \
                    --nsample 3000 \
                    --seed 42 \
                    --output_dir /content/dgm-eval/experiments_{TAG}_paperorder \
                    2>&1 | tee {RESULTS_DIR}/{TAG}_dinov2_paperorder.txt


In [ ]:
# ============== collect the numbers into one sheet (same parser as the Dec 2025 notebook) ==============
import pandas as pd, re, os

def parse(txt_path):
    metrics = {}
    if not os.path.exists(txt_path):
        return metrics
    with open(txt_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line == "" or line.lower().startswith("loading") or line.lower().startswith("saving"):
                continue
            m = re.match(r'^([A-Za-z_\-]+):\s*([-+0-9.eE]+)$', line)   # lines like  fd: 889.95
            if m:
                key, value = m.group(1), m.group(2)
                try:
                    metrics[key] = float(value)
                except ValueError:
                    metrics[key] = value
    return metrics

runs = {
    "A_dinov2_documented_order": f"{RESULTS_DIR}/{TAG}_dinov2.txt",
    "B_inception_documented_order": f"{RESULTS_DIR}/{TAG}_inception.txt",
    "C_dinov2_paper_order": f"{RESULTS_DIR}/{TAG}_dinov2_paperorder.txt",
}
table = pd.DataFrame({k: parse(v) for k, v in runs.items()})
table.index.name = "metric"
out_xlsx = f"{RESULTS_DIR}/{TAG}_metrics.xlsx"
table.to_excel(out_xlsx)
pd.set_option("display.width", 200)
print(table)
print("\nPaper columns -> DINOv2 FD = A:fd | KD = A:kd_value | FLS-Overfit = A:fls_overfit | FID = B:fd | Precision/Recall = A:precision/recall | CT = A:ct_score")
print("saved:", out_xlsx)
print("sizes used: real_train", n_real, "| real_test", n_test, "| generated", n_gen)
